[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/byu-matrix-lab/torchlingo/blob/main/docs/docs/course/lecture-08a-a8-kickoff.ipynb)

# Assignment 8 kickoff: from your cleaned corpus to a running model

**This notebook is the executable path through Assignment 8. The handout is the authority.**
Every threshold below is quoted from the handout; if the two ever disagree, the handout governs
and this notebook is wrong.

What "done for today" means: your data is split, verified and saved to Drive, and a training
run is going with checkpoints landing in Drive. The run continues after class. You do not wait
for it.

**Running in Google Colab? Two things, in this order:**

1. **Runtime → Change runtime type → GPU.** Do this *first*. Changing it later restarts the
   session and throws away everything you have run. The cell after the install will refuse to
   continue without one, on purpose: sixty epochs over 100,000 pairs on a CPU is days, not hours.
2. **Run the next two cells.** The first installs TorchLingo; the second proves the install
   worked and checks the GPU. There is nothing to uncomment.

<!-- GENERATED:purpose - change the torchlingo metadata, not this cell -->

**CS 479 course notebook** — the in-class activity for Lecture 8a (*Neural MT: Encoder-Decoder, and Why Attention Was Invented*). A head start on **A8** (due at Lecture 9).

In [ ]:
# Install TorchLingo when running in Colab. Nothing to uncomment.
#
# A commented-out install is the most reliable way to lose twenty minutes of a class: the
# cell "succeeds" because it does nothing, the next cell raises ModuleNotFoundError, and you
# cannot tell whether the library is broken or you forgot a step. So this decides for itself
# and says what it did.
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    print("Colab detected. Installing torchlingo ...")
    completed = subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", "torchlingo", "sacrebleu"],
        capture_output=True,
        text=True,
    )
    if completed.returncode != 0:
        print(completed.stdout)
        print(completed.stderr)
        raise SystemExit("pip install failed; read the output above before going on.")
    print("Installed.")
else:
    print("Not in Colab; assuming torchlingo is already installed here.")

In [ ]:
# Prove the setup worked, and refuse to continue without a GPU.
import torch
from torchlingo.config import Config
from torchlingo.data_processing import NMTDataset, collate_fn
from torchlingo.models import SimpleTransformer
from torchlingo.training import train_model
from torchlingo.training_checkpoint import TrainingCheckpointer
from torchlingo.inference import translate_batch
from torchlingo.diagnostics import check_contamination, uniform_loss

if not torch.cuda.is_available():
    raise SystemExit(
        "No GPU. Runtime > Change runtime type > GPU, then Runtime > Restart, then start again "
        "from the top. Do not train this assignment on a CPU: it will not finish this week."
    )

device = torch.device("cuda")
print("Imports OK. GPU:", torch.cuda.get_device_name(0))

## The numbers, and where they come from

These are the Assignment 8 handout's thresholds, copied here once so the rest of the notebook
reads them from one place. **If the handout changes, change this cell; nothing else in the
notebook holds a number.**

In [ ]:
# From the Assignment 8 handout ("What To Do"). The handout governs; this cell follows it.
TRAIN_FLOOR = 100_000   # at least this many training pairs, or all of it if you have less
VAL_SIZE    = 2_000
TEST_SIZE   = 2_000
MAX_TOKENS  = 100       # drop a pair if either side is longer than this; for now a token is a word
EPOCHS      = 65        # the handout says 60 to 70

SEED = 479              # so a re-run gives you the same split, and the same test set

## Step 0: mount Drive and point at your Assignment 5 corpus

Edit the two paths. They are the two sentence-aligned files you submitted for Assignment 5:
one line per segment, line *n* of one file translating line *n* of the other.

In [ ]:
from pathlib import Path
import pandas as pd

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = Path("/content/drive/MyDrive")
else:
    DRIVE = Path(".")

# TODO: your two Assignment 5 files. English on the source side; your language on the target.
SRC_FILE = DRIVE / "CS479" / "cleaned.en.txt"       # <- edit
TGT_FILE = DRIVE / "CS479" / "cleaned.xx.txt"       # <- edit

# Everything this notebook writes goes here, so you can find it and so a dropped session loses
# nothing.
OUT_DIR = DRIVE / "CS479" / "assignment8"
OUT_DIR.mkdir(parents=True, exist_ok=True)

with open(SRC_FILE, encoding="utf-8") as f:
    src_lines = [l.rstrip("\n") for l in f]
with open(TGT_FILE, encoding="utf-8") as f:
    tgt_lines = [l.rstrip("\n") for l in f]

if len(src_lines) != len(tgt_lines):
    raise SystemExit(
        f"Line counts differ: {len(src_lines):,} source, {len(tgt_lines):,} target. "
        "Those files are not sentence-aligned. Fix Assignment 5 before going on."
    )

pairs = pd.DataFrame({"src": src_lines, "tgt": tgt_lines})
pairs = pairs[(pairs.src.str.strip() != "") & (pairs.tgt.str.strip() != "")].reset_index(drop=True)
print(f"Loaded {len(pairs):,} pairs.")
if len(pairs) < TRAIN_FLOOR + VAL_SIZE + TEST_SIZE:
    print(f"That is below {TRAIN_FLOOR + VAL_SIZE + TEST_SIZE:,}, so the handout's low-resource "
          "clause applies to you: use all of it, and say so in your write-up.")

## Step 1: cap sentence length, *before* splitting

The cap removes pairs. If you split first and cap afterwards, a training set of exactly
100,000 pairs quietly becomes fewer than 100,000, and you no longer meet the floor you think you
meet. So the cap goes first, and the split sizes you see below are the sizes you submit.

Why cap at all: peak memory in training is set by the *longest* sentence in a batch, not the
typical one. Lecture 8a's numbers were 9.6 GiB held with the cap against 35.8 GiB without, for
1.3% of the pairs. Your percentage is printed below; it is your number, not the deck's.

In [ ]:
src_len = pairs.src.str.split().str.len()
tgt_len = pairs.tgt.str.split().str.len()

keep = (src_len <= MAX_TOKENS) & (tgt_len <= MAX_TOKENS)
dropped = int((~keep).sum())
print(f"Longest source: {src_len.max()} tokens. Longest target: {tgt_len.max()} tokens.")
print(f"Cap at {MAX_TOKENS}: dropping {dropped:,} pairs ({100*dropped/len(pairs):.2f}%).")

pairs = pairs[keep].reset_index(drop=True)
print(f"{len(pairs):,} pairs remain.")

## Step 2: deduplicate on the source side

Not on the pair: on the source. Two identical source sentences with different translations are
still the same source, and if one lands in training and the other in test, your test set
contains sentences the model was trained on. Lecture 7 and Lecture 8a both covered why that
number then looks like progress.

In [ ]:
before = len(pairs)
pairs = pairs.drop_duplicates(subset="src", keep="first").reset_index(drop=True)
removed = before - len(pairs)
print(f"Removed {removed:,} pairs with a repeated source ({100*removed/before:.2f}%).")
print(f"{len(pairs):,} unique-source pairs remain.")

## Step 3: split, with exact counts, seeded

The handout asks for 2,000 validation and 2,000 test pairs and everything else in training.
`torchlingo.preprocessing.split_data` takes *ratios*, so the split is written out here instead.
It is seeded: run this cell twice and you get the same test set both times, which is the only
way your BLEU tomorrow is comparable with your BLEU today.

In [ ]:
import numpy as np

rng = np.random.default_rng(SEED)
order = rng.permutation(len(pairs))

test  = pairs.iloc[order[:TEST_SIZE]].reset_index(drop=True)
val   = pairs.iloc[order[TEST_SIZE:TEST_SIZE + VAL_SIZE]].reset_index(drop=True)
train = pairs.iloc[order[TEST_SIZE + VAL_SIZE:]].reset_index(drop=True)

print(f"train {len(train):,}   val {len(val):,}   test {len(test):,}")
if len(train) < TRAIN_FLOOR:
    print(f"Training set is below the {TRAIN_FLOOR:,} floor. That is allowed for low-resource "
          "languages: use all of it, and state the count in your write-up.")

## Step 4: verify. Do not trust.

After source-side deduplication, no split *can* put one source in two sets. So this check
should pass, and the point of running it is that you saw it pass on your data rather than
assumed it. It raises rather than prints: a warning you can scroll past is not a check, and a
contaminated split here is carried into Assignments 9, 13 and 14.

In [ ]:
for name, held_out in (("test", test), ("val", val)):
    result = check_contamination(held_out.src, train.src)
    print(f"{name}: {result.detail}")
    if not result:
        raise SystemExit(
            f"The {name} set shares sources with training. Re-run from Step 2 in order; if it "
            "still fails, ask the teaching staff before training anything."
        )

overlap = set(test.src) & set(val.src)
if overlap:
    raise SystemExit(f"{len(overlap)} sources appear in both val and test. Re-run from Step 3.")
print("Clean. Three disjoint sets.")

## Step 5: write the files to Drive

Two formats, because they serve two readers. TorchLingo reads one TSV per split with `src` and
`tgt` columns. The handout asks you to submit the splits as parallel text, source and target,
which is six files. Both are written here so nothing has to be reconstructed later.

In [ ]:
for name, frame in (("train", train), ("val", val), ("test", test)):
    frame.to_csv(OUT_DIR / f"{name}.tsv", sep="\t", index=False)
    (OUT_DIR / f"{name}.src").write_text("\n".join(frame.src) + "\n", encoding="utf-8")
    (OUT_DIR / f"{name}.tgt").write_text("\n".join(frame.tgt) + "\n", encoding="utf-8")

for p in sorted(OUT_DIR.iterdir()):
    print(f"{p.name:12} {p.stat().st_size/1e6:7.2f} MB")

## Step 6: the model

The course configuration: `d_model` 256, 8 heads, 3 encoder and 3 decoder layers, feed-forward
1024. About 11.7 million parameters, and the notebook prints the exact number. Same
architecture as the library default; half the width and depth.

`num_workers=0` is set explicitly. The library default is 4, and on Colab that costs about
twenty seconds of fixed overhead per loader for no gain at any corpus size we measured.

In [ ]:
from torch.utils.data import DataLoader

config = Config(
    d_model=256,
    n_heads=8,
    num_encoder_layers=3,
    num_decoder_layers=3,
    d_ff=1024,
    batch_size=64,
    num_workers=0,
)

train_ds = NMTDataset(OUT_DIR / "train.tsv", config=config)
val_ds   = NMTDataset(OUT_DIR / "val.tsv",  src_vocab=train_ds.src_vocab, tgt_vocab=train_ds.tgt_vocab, config=config)
test_ds  = NMTDataset(OUT_DIR / "test.tsv", src_vocab=train_ds.src_vocab, tgt_vocab=train_ds.tgt_vocab, config=config)

train_loader = DataLoader(train_ds, batch_size=config.batch_size, shuffle=True,  collate_fn=collate_fn, num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=config.batch_size, shuffle=False, collate_fn=collate_fn, num_workers=0)

torch.manual_seed(SEED)
model = SimpleTransformer(
    src_vocab_size=len(train_ds.src_vocab),
    tgt_vocab_size=len(train_ds.tgt_vocab),
    config=config,
).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f"Source vocab {len(train_ds.src_vocab):,}   target vocab {len(train_ds.tgt_vocab):,}")
print(f"Model: {n_params:,} parameters")
print(f"Batches per epoch: {len(train_loader):,}   (x {EPOCHS} epochs)")

## The number to compare your first loss against

Lecture 7 derived it: a model that has learned nothing spreads probability evenly, so its
cross-entropy is exactly `ln(V)` for a target vocabulary of size `V`. That is the reference
point. Near it, nothing is learning. Falling below it, something is.

Two things to expect, so neither looks like a failure. The first `log_every` line is the mean of
the first fifty batches, and it usually sits *a little above* `ln(V)`: a freshly built model is
not perfectly uniform, and leaning the wrong way costs more than not leaning at all. And
`train_losses[0]` is the mean of the whole first epoch, by the end of which the model is well
below `ln(V)`. Compare the first logged line with the number here.

In [ ]:
V = len(train_ds.tgt_vocab)
print(f"ln(V) for V = {V:,} is {uniform_loss(V):.3f}")
print("Your first logged loss should be near this, often a little above. Your epoch-1 average will be below it.")

## Step 7: start training. This continues after class.

Three things this cell does that the handout requires:

- **`val_loader` and `save_dir`.** The trainer then keeps the checkpoint with the lowest
  validation loss, which is not always the last one. That is what your 2,000 validation pairs
  are for.
- **A `TrainingCheckpointer`**, which saves resumable state to Drive every ten minutes. If the
  session drops, run this cell again and it picks up where it stopped. Without it, a
  disconnect costs the whole run.
- **`log_every`**, so you see the start of training and a smooth curve rather than one number
  per epoch.

Sixty-five epochs at 100,000 pairs took just under two hours on a fast local machine. Nobody has
measured Colab yet. Assume worse, start it now, and leave it running when class ends.

In [ ]:
checkpointer = TrainingCheckpointer(experiment_name="assignment-8")   # Drive, in Colab
BEST_DIR = OUT_DIR / "best"

result = train_model(
    model,
    train_loader,
    val_loader=val_loader,
    num_epochs=EPOCHS,
    device=device,
    config=config,
    gradient_clip=1.0,
    save_dir=BEST_DIR,
    log_every=50,
    checkpointer=checkpointer,
)

print(f"Epochs run: {len(result.train_losses)}")
print(f"Final train loss {result.train_losses[-1]:.4f}   final val loss {result.val_losses[-1]:.4f}")
print(f"Best checkpoint: {result.best_checkpoint}")

## Before you leave the room

- The cell above is running and has printed at least one `log_every` line.
- `OUT_DIR` in Drive has nine files: three `.tsv`, three `.src`, three `.tgt`.
- You wrote down your first logged loss and `ln(V)` next to each other.

If the session drops later: open this notebook, run everything down to and including Step 6
(it is fast; nothing retrains), then run Step 7 again. The checkpointer resumes.

---

## After it finishes: translate the test set and score it

Not for today. Come back to this when Step 7 has completed. Greedy decoding, the whole test set
at once, and SacreBLEU over all 2,000 sentences, the same rule as Assignment 6. Say in your
write-up that you decoded greedily; Lecture 10 explains why that sentence matters.

In [ ]:
import sacrebleu
from torchlingo.checkpoint import load_checkpoint

# The best-by-validation model, not the last one. load_checkpoint returns a dict; the
# model is restored from its state dict.
if result.best_checkpoint:
    ckpt = load_checkpoint(result.best_checkpoint, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])

model.eval()
sources    = list(test.src)
references = list(test.tgt)

hypotheses = []
for start in range(0, len(sources), 64):
    hypotheses.extend(translate_batch(
        model, sources[start:start + 64],
        train_ds.src_vocab, train_ds.tgt_vocab,
        decode_strategy="greedy", device=device, config=config,
    ))

# One stream of hypotheses, one list of reference streams. Per-sentence nesting is the
# silent mistake Lecture 6 warned about.
bleu = sacrebleu.corpus_bleu(hypotheses, [references])
print(bleu)

(OUT_DIR / "test.hyp").write_text("\n".join(hypotheses) + "\n", encoding="utf-8")
for s, r, h in list(zip(sources, references, hypotheses))[:5]:
    print(f"\nSRC {s}\nREF {r}\nHYP {h}")